# 🩺 Amar Doctor V1 — AI Video & Neural Voice Sandbox (Colab GPU)
### Free-tier AI telemedicine backend running on Google Colab
- 🎙️ **Edge-TTS Bengali Neural Voice Synthesis** (`bn-BD-NabanitaNeural` / `bn-BD-PradeepNeural`)
- 🧠 **faster-whisper `large-v3` Bengali speech recognition (STT)**
- 🤖 **Groq (GPT-OSS-120B) medical triage engine**
- 📹 **MuseTalk GPU lip-sync avatar (optional)** — real inpainted video, not a looping clip
- 🌐 **Free Cloudflare Tunnel** for a public HTTPS URL your local Next.js app can call

**Two independent pieces, and you only need the first one:**

| | Needs | Cells |
|---|---|---|
| 🎙️ Voice calls | a T4 (or better) GPU + a Groq API key | 1, 2, 3, 6 |
| 📹 Video calls with real GPU lip-sync | the above, **plus** the MuseTalk environment and a doctor portrait/clip | all |

Skip the MuseTalk cells (or set `ENABLE_MUSETALK = "0"` in cell 3) and everything still works —
video calls just show an audio-reactive avatar instead of GPU-rendered lip-sync, exactly like
running the backend locally without the sidecar.

**Everything installs to Colab's local disk (`/content`), nothing to Google Drive.** That keeps the
install fast and avoids Drive's slow FUSE mount — the tradeoff is that a runtime reset wipes it, so
cell 5 re-downloads ~7.3GB of weights next session (roughly 10 min on Colab's network). See
[`MUSETALK_SETUP.md`](../MUSETALK_SETUP.md) for the same recipe on Windows.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Clone your repository
!git clone https://github.com/CHANGE-ME/amar-doctor.git /content/amar-doctor  # <-- set your repo URL
%cd /content/amar-doctor

## 3. Configure this session
`GROQ_API_KEY` is required for every mode — without it every reply is a canned fallback that
ignores what the patient said. `ENABLE_MUSETALK` controls whether `colab_runner.py` (cell 6)
starts the video renderer; leave it `"1"` if you're running the MuseTalk cells below, or set it
to `"0"` to start only the voice pipeline.

In [ ]:
# 3. Session secrets and toggles
import os

os.environ["GROQ_API_KEY"] = "your_groq_key_here"  # https://console.groq.com/keys
os.environ["ENABLE_MUSETALK"] = "1"  # "0" to skip GPU lip-sync and start voice-only

# Optional: a smaller/larger Whisper checkpoint.
# os.environ["WHISPER_MODEL_SIZE"] = "large-v3"  # see backend/README.md before changing this

## 4. Doctor avatar asset (needed for video calls)
MuseTalk repaints a **photographic** mouth region — it cannot work from an illustration, and needs
a real portrait or short clip. See [`backend/static/AVATAR.md`](../backend/static/AVATAR.md) for the
framing requirements and the licence note it asks you to keep.

Prefer a **5–10s clip of the person sitting still** (`doctor_idle_source.mp4`) over a single still
image — MuseTalk cycles through prepared frames, and a still gives a frozen head with a moving jaw.

Skip this cell if `ENABLE_MUSETALK = "0"`. You'll re-upload after a runtime reset, same as the
weights.

In [ ]:
# 4. Get a doctor portrait/clip into backend/static/
from pathlib import Path

STATIC_DIR = Path("backend/static")
STATIC_DIR.mkdir(parents=True, exist_ok=True)

wanted = ["doctor_idle_source.mp4", "doctor_avatar.png"]
have = [n for n in wanted if (STATIC_DIR / n).exists()]

if have:
    print(f"Already present: {have}")
else:
    print("Upload doctor_idle_source.mp4 (preferred) or doctor_avatar.png:")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = STATIC_DIR / name
        dest.write_bytes(data)
        print(f"Saved {dest}")

## 5. Install MuseTalk
Isolated on purpose: MuseTalk needs `mmcv`/`mmpose`/`mmdet` and pins `numpy==1.23.5`, which would
break the `faster-whisper` speech recognition running in Colab's own Python. So this installs into
its own Python 3.10 virtualenv, same as the local Windows recipe in `MUSETALK_SETUP.md` — Colab's
own Python is never touched.

**Time:** ~10 min, most of it the 7.3GB weight download. Everything lands on `/content`, so a
runtime reset means running this again.

Safe to re-run within a session: every step checks whether it already did its work and skips, so a
dropped connection mid-install just needs this cell run again rather than starting over.

**If `mmcv` fails to find a wheel:** Colab's Python/CUDA combo has drifted from what mmcv 2.0.1
publishes prebuilt wheels for. The error names the missing wheel explicitly — that's a real risk of
pinning to a research repo's exact versions, not a bug in this script. `MUSETALK_SETUP.md`'s "If
mmcv will not build" section describes the fallback.

In [ ]:
%%bash
set -e

# Colab presets UV_SYSTEM_PYTHON=1, which makes `uv venv` emit a confusing
# warning and can aim later installs at the system Python instead of ours.
unset UV_SYSTEM_PYTHON

# Everything on Colab's local disk. Fast, but wiped by a runtime reset.
MUSETALK_ROOT="/content/MuseTalk"
MUSETALK_VENV="/content/musetalk-venv"

echo "MuseTalk repo : $MUSETALK_ROOT"
echo "MuseTalk venv : $MUSETALK_VENV"

# ffmpeg -- Colab images ship it via apt already; this is just a safety net.
command -v ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)
echo "ffmpeg: $(command -v ffmpeg)"

# 1. Isolated Python 3.10.
#    --seed is REQUIRED: without it uv creates a venv containing no pip at
#    all, and both `python -m pip` and `mim` (which shells out to pip) die
#    with "No module named pip". It also seeds setuptools and wheel, which
#    mim and chumpy respectively need below.
pip install -q uv
if [ ! -x "$MUSETALK_VENV/bin/python" ]; then
  echo "--- Creating Python 3.10 venv ---"
  uv python install 3.10
  uv venv --python 3.10 --seed "$MUSETALK_VENV"
else
  echo "--- venv already exists, skipping ---"
fi
PY="$MUSETALK_VENV/bin/python"
"$PY" -m pip --version

# 2. Clone MuseTalk.
if [ ! -d "$MUSETALK_ROOT/.git" ]; then
  echo "--- Cloning MuseTalk ---"
  git clone --depth 1 https://github.com/TMElyralab/MuseTalk "$MUSETALK_ROOT"
else
  echo "--- MuseTalk repo already cloned, skipping ---"
fi

# 3. torch + the mmcv/mmdet/mmpose stack, in dependency order (mim resolves
#    mmcv against whichever torch is already installed).
if ! "$PY" -c "import mmpose" >/dev/null 2>&1; then
  echo "--- Installing torch 2.0.1+cu118 ---"
  "$PY" -m pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
  "$PY" -m pip install -q -U openmim
  # mim (unmaintained) needs pkg_resources, which setuptools >=70 dropped --
  # and --seed above installs a modern one, so pin it back down. Both this
  # and chumpy's missing-wheel failure bit the local Windows install too;
  # see MUSETALK_SETUP.md.
  "$PY" -m pip install -q "setuptools<70" wheel
  echo "--- mim install mmengine ---"
  "$PY" -m mim install mmengine
  echo "--- mim install mmcv==2.0.1 ---"
  "$PY" -m mim install "mmcv==2.0.1"
  echo "--- chumpy (mmpose dependency with a broken build-isolation setup.py) ---"
  "$PY" -m pip install -q --no-build-isolation chumpy
  echo "--- mim install mmdet==3.1.0 ---"
  "$PY" -m mim install "mmdet==3.1.0"
  echo "--- mim install mmpose==1.1.0 ---"
  "$PY" -m mim install "mmpose==1.1.0"
else
  echo "--- mmcv/mmdet/mmpose already installed, skipping ---"
fi

# 4. MuseTalk's own requirements, minus training-only/UI extras we don't run headless.
if ! "$PY" -c "import diffusers" >/dev/null 2>&1; then
  echo "--- Installing MuseTalk's own requirements (headless) ---"
  grep -vE '^(tensorflow|tensorboard|gradio)' "$MUSETALK_ROOT/requirements.txt" > "$MUSETALK_ROOT/requirements-headless.txt"
  "$PY" -m pip install -q -r "$MUSETALK_ROOT/requirements-headless.txt"
  "$PY" -m pip install -q "huggingface_hub[cli]==0.30.2" fastapi "uvicorn[standard]" httpx
else
  echo "--- MuseTalk requirements already installed, skipping ---"
fi

# 5. Weights (~7.3GB), straight into the repo's own models/ dir. Default HF
#    endpoint -- MuseTalk's own download script points at a China mirror;
#    deliberately not used here.
WEIGHTS_DIR="$MUSETALK_ROOT/models"
CLI="$MUSETALK_VENV/bin/huggingface-cli"
if [ ! -f "$WEIGHTS_DIR/musetalkV15/unet.pth" ]; then
  echo "--- Downloading weights (~7.3GB) ---"
  "$CLI" download TMElyralab/MuseTalk       --local-dir "$WEIGHTS_DIR"
  "$CLI" download yzd-v/DWPose              --local-dir "$WEIGHTS_DIR/dwpose"  --include "dw-ll_ucoco_384.pth"
  "$CLI" download stabilityai/sd-vae-ft-mse --local-dir "$WEIGHTS_DIR/sd-vae"  --include "config.json" "diffusion_pytorch_model.bin"
  "$CLI" download openai/whisper-tiny       --local-dir "$WEIGHTS_DIR/whisper" --include "config.json" "pytorch_model.bin" "preprocessor_config.json"
  "$CLI" download ManyOtherFunctions/face-parse-bisent --local-dir "$WEIGHTS_DIR/face-parse-bisent" --include "79999_iter.pth" "resnet18-5c106cde.pth"
else
  echo "--- Weights already present, skipping download ---"
fi

echo ""
echo "MuseTalk environment ready."
du -sh "$WEIGHTS_DIR" 2>/dev/null || true
df -h /content | tail -1


## 6. Launch backend + (if installed) MuseTalk + tunnel
`colab_runner.py` installs the main backend's own dependencies, starts the MuseTalk renderer in
the background **only if** cells 4 and 5 actually completed (otherwise it says so and continues
without video), starts the FastAPI backend, and opens the Cloudflare tunnel. Watch for:

- `✓ MuseTalk renderer ready` — video calls get real GPU lip-sync
- `ℹ️ ... skipping GPU lip-sync` — audio-reactive avatar instead; it names which cell to check

Then copy the `https://....trycloudflare.com` URL from the output and paste it into the "Connect
Backend" box on the `/chat` page of your locally-running Next.js app.

In [ ]:
# 6. Launch backend & Cloudflare tunnel
!python3 backend/colab_runner.py